In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

# 1. Carga de datos
df_gei = pd.read_csv(r'/content/drive/MyDrive/UAIN1008R_Proyecto/datos/processed/Limpio - Dataset GEI CSV_0.csv')
df_trafico = pd.read_csv(r'/content/drive/MyDrive/UAIN1008R_Proyecto/datos/processed/Limpio - Información Trafico - Carga y Operaciones - LAP Ene 2019 - Dic 2025 - Organismo Supervisor de la Inversión en Infraestructura de Transporte de Uso Público – OSITRAN.csv')

# Pre-procesamiento de fechas
df_gei['ANIO_DATE'] = pd.to_datetime(df_gei['ANIO'])
df_trafico['FECHA'] = pd.to_datetime(df_trafico['FECHA'])
df_trafico['ANIO'] = df_trafico['FECHA'].dt.year

print("--- Dataset GEI ---")
print(df_gei.head())
print("\n--- Dataset Tráfico ---")
print(df_trafico.head())

# Extraer el año si viene como fecha completa y asegurar que sea numérico
df_gei['ANIO'] = pd.to_datetime(df_gei['ANIO']).dt.year

Mounted at /content/drive
--- Dataset GEI ---
  FECHA_CORTE        ANIO    SECTOR                              CATEGORIA  \
0  2024-05-21  2019-01-01  ENERGIA   ACTIVIDADES DE QUEMA DE COMBUSTIBLES    
1  2024-05-21  2019-01-01  ENERGIA   ACTIVIDADES DE QUEMA DE COMBUSTIBLES    
2  2024-05-21  2019-01-01  ENERGIA   ACTIVIDADES DE QUEMA DE COMBUSTIBLES    
3  2024-05-21  2019-01-01  ENERGIA   ACTIVIDADES DE QUEMA DE COMBUSTIBLES    
4  2024-05-21  2019-01-01  ENERGIA   ACTIVIDADES DE QUEMA DE COMBUSTIBLES    

                                     SUBCATEGORIA  \
0                           INDUSTRIAS DE ENERGIA   
1                           INDUSTRIAS DE ENERGIA   
2                           INDUSTRIAS DE ENERGIA   
3  INDUSTRIAS MANUFACTURERAS Y DE LA CONSTRUCCION   
4  INDUSTRIAS MANUFACTURERAS Y DE LA CONSTRUCCION   

                                   FUENTE_DE_EMISION IPCC_CLASF  \
0  PRODUCCION DE ELECTRICIDAD Y CALOR COMO ACTIVI...       1A1a   
1                             RE

In [2]:
# 2. Preparación para KPIs
# GEI: agrupar TODOS los sectores por año, filtrar 2000-2019
gei_anual = df_gei.groupby('ANIO')['EMISIONES_GEI_GGCO2EQ'].sum().reset_index()
gei_anual.columns = ['ANIO', 'TOTAL_GEI']
gei_anual = gei_anual.sort_values('ANIO').reset_index(drop=True)

print("--- GEI por año (2000-2019) ---")
print(gei_anual)

# Tráfico: agrupar por año
trafico_anual = df_trafico.groupby('ANIO')[['TOTAL CARGA NACIONAL', 'TOTAL CARGA INTERNACIONA']].sum().reset_index()
trafico_anual['CARGA_TOTAL'] = trafico_anual['TOTAL CARGA NACIONAL'] + trafico_anual['TOTAL CARGA INTERNACIONA']

print("\n--- Tráfico por año ---")
trafico_anual = trafico_anual.round(3)
print(trafico_anual)

--- GEI por año (2000-2019) ---
   ANIO  TOTAL_GEI
0  2000   67993.53
1  2005   74950.74
2  2010  183400.26
3  2012  189342.11
4  2014  188744.57
5  2016  204252.82
6  2019  210404.41

--- Tráfico por año ---
   ANIO  TOTAL CARGA NACIONAL  TOTAL CARGA INTERNACIONA  CARGA_TOTAL
0  2019             27324.773                244001.378   271326.151
1  2020             18420.144                171944.917   190365.061
2  2021             22797.150                196405.687   219202.837
3  2022             25197.825                194034.382   219232.207
4  2023             27610.049                183593.347   211203.396
5  2024             27798.061                201164.958   228963.019
6  2025             30736.460                236416.688   267153.148


# Calculo de KPI

In [3]:
# Unir datos para cálculo de KPIs (solo año 2019 coincide entre ambos datasets)
df_kpi = pd.merge(gei_anual, trafico_anual, on='ANIO')

# KPI 1: Intensidad de Emisión por Carga (IEC) = emisiones / carga total
df_kpi['IEC'] = df_kpi['TOTAL_GEI'] / df_kpi['CARGA_TOTAL']

print("--- KPI Intensidad de Emisión por Carga (IEC) ---")
print(df_kpi[['ANIO', 'TOTAL_GEI', 'CARGA_TOTAL', 'IEC']])

# KPI 2: Variación YoY de Emisiones
gei_anual['Variacion_Pct'] = gei_anual['TOTAL_GEI'].pct_change() * 100
print("\n--- Evolución Histórica de Emisiones GEI ---")
print(gei_anual)

--- KPI Intensidad de Emisión por Carga (IEC) ---
   ANIO  TOTAL_GEI  CARGA_TOTAL       IEC
0  2019  210404.41   271326.151  0.775467

--- Evolución Histórica de Emisiones GEI ---
   ANIO  TOTAL_GEI  Variacion_Pct
0  2000   67993.53            NaN
1  2005   74950.74      10.232165
2  2010  183400.26     144.694395
3  2012  189342.11       3.239826
4  2014  188744.57      -0.315587
5  2016  204252.82       8.216528
6  2019  210404.41       3.011753


In [4]:
# 3. ANÁLISIS TEMPORAL (YoY)
gei_anual['Variacion_Pct'] = gei_anual['TOTAL_GEI'].pct_change() * 100

print("--- Evolución Histórica de Emisiones GEI (2000-2019) ---")
print(gei_anual.round(3))

--- Evolución Histórica de Emisiones GEI (2000-2019) ---
   ANIO  TOTAL_GEI  Variacion_Pct
0  2000   67993.53            NaN
1  2005   74950.74         10.232
2  2010  183400.26        144.694
3  2012  189342.11          3.240
4  2014  188744.57         -0.316
5  2016  204252.82          8.217
6  2019  210404.41          3.012


In [5]:
# 4. SEGMENTACIÓN
# Segmentación 1: Emisiones por Gas (Para entender qué contamina más)
seg_gas = df_gei.groupby('ANIO')[['METANO_EQUIVALENTE_GGCO2EQ', 'OXIDO_NITROSO_EQUIVALENTE_GGCO2EQ']].sum().tail().reset_index()

# Segmentación 2: Carga por Tipo de Tráfico (Aéreo)
seg_carga = df_trafico.groupby('ANIO', as_index=False)[['TOTAL CARGA NACIONAL', 'TOTAL CARGA INTERNACIONA']].sum().tail()

print("\n--- Segmentación por Tipo de Gas (CO2eq) ---")
print(seg_gas)
print("\n--- Segmentación por Tipo de Carga (Toneladas) ---")
print(seg_carga)


--- Segmentación por Tipo de Gas (CO2eq) ---
   ANIO  METANO_EQUIVALENTE_GGCO2EQ  OXIDO_NITROSO_EQUIVALENTE_GGCO2EQ
0  2010                    37510.44                           10222.96
1  2012                    36671.89                           10536.22
2  2014                    38451.14                           10554.72
3  2016                    37536.38                           10583.92
4  2019                    39014.01                           11085.18

--- Segmentación por Tipo de Carga (Toneladas) ---
   ANIO  TOTAL CARGA NACIONAL  TOTAL CARGA INTERNACIONA
2  2021           22797.15037              196405.68670
3  2022           25197.82548              194034.38158
4  2023           27610.04904              183593.34681
5  2024           27798.06100              201164.95800
6  2025           30736.46000              236416.68800


# Prueba Estadística

In [6]:
from scipy import stats
import numpy as np

# KPI: Variación YoY de Emisiones
# 1. Preparación de datos históricos (Dataset GEI)
# Dividimos la serie histórica en dos décadas para ver si hubo un cambio significativo
periodo_A = gei_anual[gei_anual['ANIO'].between(2000, 2012)]['TOTAL_GEI']
periodo_B = gei_anual[gei_anual['ANIO'].between(2014, 2019)]['TOTAL_GEI']

print(f"Análisis Estadístico: Evolución Histórica de Emisiones GEI")
print("="*60)
print(f"Década 2000-2012 (Muestras: {len(periodo_A)} años)")
print(f"Década 2014-2019 (Muestras: {len(periodo_B)} años)")
print("-"*60)

# 2. Verificación de normalidad (Requisito para elegir la prueba)
# Si p > 0.05, los datos siguen una distribución normal
shapiro_A = stats.shapiro(periodo_A)
shapiro_B = stats.shapiro(periodo_B)

print(f"Normalidad Década 1: p = {shapiro_A.pvalue:.4f}")
print(f"Normalidad Década 2: p = {shapiro_B.pvalue:.4f}")

# 3. Prueba estadística de Hipótesis
# H0: No hay diferencia significativa entre las emisiones de ambas décadas.
# H1: Las emisiones han cambiado significativamente (crecimiento o reducción).

if shapiro_A.pvalue > 0.05 and shapiro_B.pvalue > 0.05:
    print("\nMétodo: Prueba t de Student (Paramétrica)")
    stat, p_value = stats.ttest_ind(periodo_A, periodo_B, equal_var=False)
else:
    print("\nMétodo: Prueba de Mann-Whitney U (No paramétrica)")
    stat, p_value = stats.mannwhitneyu(periodo_A, periodo_B, alternative='two-sided')

# 4. Resultados e Interpretación
print(f"Estadístico de prueba: {stat:.4f}")
print(f"P-value: {p_value:.4f}")
print("-"*60)

if p_value < 0.05:
    mean_diff = ((periodo_B.mean() - periodo_A.mean()) / periodo_A.mean()) * 100
    print("RESULTADO: → SE RECHAZA LA HIPÓTESIS NULA (H₀)")
    print(f"Interpretación: Existe una diferencia estadística significativa entre periodos.")
    print(f"Análisis: Las emisiones promedio aumentaron un {mean_diff:.2f}% en la última década.")
else:
    print("RESULTADO: → NO SE RECHAZA LA HIPÓTESIS NULA (H₀)")
    print("Interpretación: No hay evidencia suficiente para afirmar un cambio significativo.")

Análisis Estadístico: Evolución Histórica de Emisiones GEI
Década 2000-2012 (Muestras: 4 años)
Década 2014-2019 (Muestras: 3 años)
------------------------------------------------------------
Normalidad Década 1: p = 0.0669
Normalidad Década 2: p = 0.5332

Método: Prueba t de Student (Paramétrica)
Estadístico de prueba: -2.1339
P-value: 0.1163
------------------------------------------------------------
RESULTADO: → NO SE RECHAZA LA HIPÓTESIS NULA (H₀)
Interpretación: No hay evidencia suficiente para afirmar un cambio significativo.
